# Notebook 07: Grid Convergence, Energy Budget & Monte Carlo Tamper Detection

Three Phase 1 analyses in one notebook:

1. **3D Grid Convergence** — Scale N from 5→40, measure frequency/damping error, estimate convergence order
2. **CMOS Interface Energy Budget** — Full write/read energy model, technology comparison, kill criteria
3. **Monte Carlo Tamper Detection** — Statistical confidence on dilatancy-based tamper signal
4. **Auto-Validated Claims Checklist** — Full claims table populated from live experiments

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from simulations.convergence import (
    run_convergence_study, stressed_convergence_study, quick_convergence_check,
)
from simulations.cmos_interface import (
    CMOSInterfaceModel, compute_energy_budget,
    technology_comparison, format_comparison_table,
    energy_kill_check, TECH_NODES, excitation_energy,
)
from experiments.exp03_dilatancy_tamper import (
    monte_carlo_tamper_detection, summarize_mc,
)
from analysis.comparison import run_all_validations, validation_summary

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print('All modules loaded.')

All modules loaded.


## 1. 3D Grid Convergence Study

The Phase 0 3D FDTD solver used N=5 interior points (125 DOFs). We now scale to N=40 (64,000 DOFs) and measure convergence.

In [ ]:
# Run convergence study with long time window for FFT resolution
# Δf = 1/t_max, so t_max=0.2 gives Δf=5 Hz (1.7% at f≈294 Hz)
# Need t_max ~ 1.0s for sub-percent FFT resolution at these frequencies
study = run_convergence_study(
    N_values=[5, 8, 10, 15, 20, 25],
    L=1.0, c=340.0, eta=50.0,
    t_max=0.5, n_points=10000,  # Δf ≈ 2 Hz → 0.7% resolution
)
print(study.summary_table())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Frequency error vs N
ax = axes[0]
ax.loglog(study.N_values, study.freq_errors, 'bo-', lw=2, ms=8)
# Expected O(h²) slope
N_ref = study.N_values.astype(float)
h2_line = study.freq_errors[0] * (N_ref[0]/N_ref)**2
ax.loglog(N_ref, h2_line, 'r--', alpha=0.6, label='O(h²) reference')
ax.set_xlabel('N (grid points per dim)')
ax.set_ylabel('Frequency Error (%)')
ax.set_title('Frequency Convergence')
ax.legend()
ax.grid(True, alpha=0.3)

# Damping error vs N
ax = axes[1]
ax.semilogy(study.N_values, study.eta_errors, 'go-', lw=2, ms=8)
ax.set_xlabel('N (grid points per dim)')
ax.set_ylabel('Damping Rate Error (%)')
ax.set_title('Damping Convergence')
ax.grid(True, alpha=0.3)

# Wall time scaling
ax = axes[2]
ax.loglog(study.dofs_array, study.wall_times, 'rs-', lw=2, ms=8)
# Expected O(N³) ~ O(DOFs) scaling for ODE setup, O(DOFs²) for solve
dofs_ref = study.dofs_array.astype(float)
t_ref_line = study.wall_times[0] * (dofs_ref/dofs_ref[0])**1.5
ax.loglog(dofs_ref, t_ref_line, 'k--', alpha=0.4, label='O(DOFs^1.5)')
ax.set_xlabel('DOFs (N³)')
ax.set_ylabel('Wall Time (s)')
ax.set_title('Computational Cost')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle(f'3D FDTD Convergence — Est. Order: {study.estimated_order:.2f}', fontsize=14)
plt.tight_layout()
plt.savefig('../analysis/figures/convergence_study.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nRichardson-extrapolated frequency: {study.richardson_f:.6f} Hz')
print(f'Analytical frequency:              {study.points[-1].f_theory:.6f} Hz')
print(f'Final grid error:                  {study.freq_errors[-1]:.4f}%')

## 2. CMOS Interface Energy Budget

Can WCFOMA actually achieve fJ-scale operations through a realistic CMOS interface?

In [ ]:
# Default model: 28nm CMOS, inductive sensing
budget = compute_energy_budget()
print(budget.summary())
print()

# Kill criteria check
checks = energy_kill_check(budget)
print('Kill Criteria:')
for name, (desc, passes) in checks.items():
    icon = '✅' if passes else '❌'
    print(f'  {icon} {desc}')

In [ ]:
# Energy breakdown pie chart + tech comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart of energy breakdown
ax = axes[0]
breakdown = budget.breakdown_dict()
# Filter to individual components (not totals)
components = {k: v for k, v in breakdown.items()
              if k not in ('Write total', 'Read total', 'Total', 'Thermal min')}
labels = list(components.keys())
values = np.array(list(components.values()))
# Remove zero values
mask = values > 0
labels = [l for l, m in zip(labels, mask) if m]
values = values[mask]
values_fj = values * 1e15
ax.pie(values_fj, labels=[f'{l}\n({v:.1f} fJ)' for l, v in zip(labels, values_fj)],
       autopct='%1.0f%%', startangle=90)
ax.set_title(f'Energy Breakdown ({budget.E_total*1e15:.1f} fJ total)')

# Technology comparison bar chart
ax = axes[1]
techs = technology_comparison(budget)
names = list(techs.keys())
energies = [v['energy_per_op_J'] for v in techs.values()]
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']
bars = ax.barh(names, [e * 1e15 for e in energies], color=colors[:len(names)])
ax.set_xscale('log')
ax.set_xlabel('Energy per Operation (fJ)')
ax.set_title('Technology Comparison')
ax.axvline(1e3, color='red', ls='--', alpha=0.5, label='1 pJ kill line')
ax.legend()
for bar, e in zip(bars, energies):
    ax.text(bar.get_width() * 1.1, bar.get_y() + bar.get_height()/2,
            f'{e*1e15:.0f} fJ', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../analysis/figures/energy_budget.png', dpi=150, bbox_inches='tight')
plt.show()

print(format_comparison_table(techs))

In [ ]:
# Energy scaling across CMOS technology nodes
fig, ax = plt.subplots(figsize=(8, 5))

nodes = list(TECH_NODES.keys())
energies_by_node = []
for node in nodes:
    model = CMOSInterfaceModel(tech_node=node)
    b = compute_energy_budget(model)
    energies_by_node.append(b.E_total * 1e15)

ax.bar(nodes, energies_by_node, color=['#FF9800', '#4CAF50', '#2196F3', '#9C27B0'])
ax.set_ylabel('Total Energy (fJ)')
ax.set_xlabel('CMOS Technology Node')
ax.set_title('Energy Scaling with Technology Node')
ax.axhline(1000, color='red', ls='--', alpha=0.5, label='1 pJ kill line')
ax.legend()
for i, e in enumerate(energies_by_node):
    ax.text(i, e + 5, f'{e:.0f} fJ', ha='center', fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nBest case ({nodes[-1]}): {energies_by_node[-1]:.1f} fJ')
print(f'Worst case ({nodes[0]}): {energies_by_node[0]:.1f} fJ')
print(f'Paper claim: fJ range — {"CONFIRMED" if energies_by_node[-1] < 1000 else "REFUTED"}')

## 3. Monte Carlo Tamper Detection

Add measurement noise + thermal jitter to the dilatancy tamper signal and compute detection reliability.

In [ ]:
# Run Monte Carlo (5000 trials per gamma)
mc = monte_carlo_tamper_detection(
    n_trials=5000,
    noise_std_fraction=0.001,  # 0.1% measurement noise
    thermal_jitter_K=0.5,
    snr_threshold=3.0,
)
print(summarize_mc(mc))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Detection probability vs gamma
ax = axes[0]
ax.plot(mc.gamma_values, mc.detection_probability, 'b-', lw=2, label='P(detect)')
ax.fill_between(mc.gamma_values, mc.confidence_lower, mc.confidence_upper,
                alpha=0.2, color='blue', label='95% CI')
ax.axhline(0.95, color='green', ls='--', alpha=0.7, label='95% target')
ax.axhline(0.05, color='red', ls='--', alpha=0.7, label='5% FP limit')
if not np.isnan(mc.min_reliable_gamma):
    ax.axvline(mc.min_reliable_gamma, color='orange', ls=':', lw=2,
               label=f'Min reliable γ={mc.min_reliable_gamma:.3f}')
ax.set_xlabel('Shear Strain γ')
ax.set_ylabel('Detection Probability')
ax.set_title('Tamper Detection Reliability')
ax.legend(fontsize=8, loc='center right')
ax.grid(True, alpha=0.3)

# ROC-like curve (detection vs false positive at different thresholds)
ax = axes[1]
thresholds = [1.0, 2.0, 3.0, 5.0, 8.0]
for thr in thresholds:
    mc_t = monte_carlo_tamper_detection(
        n_trials=2000, snr_threshold=thr,
        noise_std_fraction=0.001, thermal_jitter_K=0.5,
    )
    ax.plot(mc_t.gamma_values, mc_t.detection_probability,
            label=f'{thr}σ threshold', lw=1.5)
ax.axhline(0.95, color='gray', ls='--', alpha=0.5)
ax.set_xlabel('Shear Strain γ')
ax.set_ylabel('Detection Probability')
ax.set_title('Sensitivity to Detection Threshold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Noise level sweep
ax = axes[2]
noise_levels = [0.0005, 0.001, 0.002, 0.005, 0.01]
for nl in noise_levels:
    mc_n = monte_carlo_tamper_detection(
        n_trials=2000, noise_std_fraction=nl,
        thermal_jitter_K=0.5, snr_threshold=3.0,
    )
    ax.plot(mc_n.gamma_values, mc_n.detection_probability,
            label=f'σ/f₀={nl*100:.1f}%', lw=1.5)
ax.axhline(0.95, color='gray', ls='--', alpha=0.5)
ax.set_xlabel('Shear Strain γ')
ax.set_ylabel('Detection Probability')
ax.set_title('Sensitivity to Measurement Noise')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle('Monte Carlo Tamper Detection Analysis (5000 trials)', fontsize=13)
plt.tight_layout()
plt.savefig('../analysis/figures/mc_tamper_detection.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'False positive rate:     {mc.false_positive_rate:.4f}')
print(f'False negative (γ=0.5):  {mc.false_negative_rate_at_05:.4f}')
print(f'Min reliable γ (>95%):   {mc.min_reliable_gamma:.4f}')

## 4. Auto-Validated Claims Checklist

Run all experiments and populate the claims table automatically.

In [ ]:
# Auto-populate claims from live experiments
print(validation_summary())